In [22]:
import torch
import json

In [49]:
class Quantizer():
    def __init__(self, q_type: str, dtype: torch.dtype):
        self.q_type = q_type
        self.dtype = dtype

    def _compute_symmetric_scale(self) -> float:
        return max(abs(self.w_min), abs(self.w_max)) / self.q_max

    def _compute_asymmetric_scale(self) -> float:
        return (self.w_max - self.w_min) / (self.q_max - self.q_min)

    def compute_scale(self) -> float:
        if self.q_type == "symmetric":
            return self._compute_symmetric_scale()

        return self._compute_asymmetric_scale()

    def compute_zero_point(self) -> int:
        return torch.round(self.q_min - (self.w_min / self.scale))

    def get_q(self):
        if self.dtype == torch.uint8:
            return 0, 255
        elif self.dtype == torch.int8:
            return -128, 127        
        elif self.dtype == torch.int4:
            return -8, -7
        elif self.dtype == torch.uint4:
            return 0, 15

    def _quantize(self, w: torch.Tensor):
        q = torch.round(w / self.scale)

        if self.q_type == "asymmetric":
            return q + self.zero_point

        q = torch.clamp(q, self.q_min, self.q_max)

        return q.to(self.dtype)

    def quantize(self, w: torch.Tensor):
        self.w_min, self.w_max = torch.min(w), torch.max(w)
        self.q_min, self.q_max = self.get_q()

        self.scale = self.compute_scale()
        self.zero_point = self.compute_zero_point()

        return self._quantize(w)

    def dequantize(self, quantized_w: torch.Tensor):
        if self.q_type == "asymmetric":
            return self.scale * (quantized_w - self.zero_point)

        return self.scale * quantized_w

    def compute_efficiency(self, q_w: torch.Tensor, d_w: torch.Tensor, w: torch.Tensor):
        delta = w - d_w

        mae = torch.mean(torch.abs(delta))
        mse = torch.mean(torch.square(delta))
        rmse = torch.sqrt(mse)
        snr = 10 * torch.log10(torch.sum(torch.square(w)) / torch.sum(torch.square(w - d_w)))

        # Model Size
        original = (w.numel() * w.element_size()) / (1024 ** 3)
        compressed = (q_w.numel() * q_w.element_size()) / (1024 ** 3)

        cr =  original / compressed
        sr = ((original - compressed) / original) * 100

        return {
            'mean_absolute_error': mae.item(),
            'mean_squared_error': mse.item(),
            'root_mean_squared_error': rmse.item(),
            'signal_to_noise_ratio': snr.item(),
            'compression_ratio': cr,
            'storage_reduction': sr
        }

In [8]:
def print_stats(w: torch.Tensor):
    print(f"Shape: {w.shape}")
    print(f"Min Value: {torch.min(w)}")
    print(f"Max Value: {torch.max(w)}")

In [50]:
w = torch.randn(3, 4, dtype=torch.float32)

print_stats(w)

Shape: torch.Size([3, 4])
Min Value: -1.6698781251907349
Max Value: 1.0285531282424927


In [52]:
quantizer = Quantizer("symmetric", dtype=torch.uint8)

q_w = quantizer.quantize(w)
print_stats(q_w)

Shape: torch.Size([3, 4])
Min Value: 0
Max Value: 157


In [53]:
d_w = quantizer.dequantize(q_w)
print_stats(d_w)

Shape: torch.Size([3, 4])
Min Value: 0.0
Max Value: 1.028120994567871


In [54]:
efficiency_metrics = quantizer.compute_efficiency(q_w, d_w, w)

print(json.dumps(efficiency_metrics, indent=4))

{
    "mean_absolute_error": 0.4399157762527466,
    "mean_squared_error": 0.5021883249282837,
    "root_mean_squared_error": 0.7086524963378906,
    "signal_to_noise_ratio": 1.420434832572937,
    "compression_ratio": 4.0,
    "storage_reduction": 75.0
}
